# safepyrun — Safe Python Sandbox for LLM Tools

**safepyrun** is an allowlist-based Python sandbox that lets LLMs execute code safely in your real environment. Instead of isolating code in a container (which cuts it off from libraries, data, and tools it needs), safepyrun runs in-process with controlled access to a curated subset of Python's stdlib.

## How It Works

```
LLM wants to run code
       |
       v
 RunPython(code)
       |
       v
 RestrictedPython compiles to modified AST
       |
       v
 Every attribute/item access goes through gatekeepers
       |
       v
 Allowlist checks: is this callable permitted?
       |
       v
 Result returned (stdout, stderr, return value)
```

## Key Features
- **Allowlist-based**: Only permitted callables are accessible
- **In-process**: Access to your real objects and data
- **Async-native**: Supports `await`, `async for`, `async with`
- **Write policies**: Controlled filesystem writes via `ok_dests`
- **State persistence**: Variables ending with `_` persist across calls

## Basic Usage

In [ ]:
from safepyrun import RunPython

# Create a sandbox instance
pyrun = RunPython()

In [ ]:
# Simple expression
await pyrun('1 + 1')

In [ ]:
# Print output is captured
await pyrun('print("Hello from the sandbox!"); 42')

## Module Access

The default allowlist covers a large subset of the standard library:

In [ ]:
# Math operations
await pyrun('import math; math.sqrt(144)')

In [ ]:
# String processing with regex
await pyrun('''
import re
text = "Hello World 123 Foo 456"
numbers = re.findall(r"\\d+", text)
f"Found numbers: {numbers}"
''')

In [ ]:
# JSON parsing
await pyrun('''
import json
data = json.loads('{"name": "dialeng", "version": "0.1.0"}')
f"Project: {data['name']} v{data['version']}"
''')

In [ ]:
# Path operations (read-only by default)
await pyrun('''
from pathlib import Path
p = Path(".")
files = sorted([f.name for f in p.iterdir() if f.is_file()])[:5]
f"First 5 files: {files}"
''')

## The `_` Suffix Convention

Variables and functions created with a trailing `_` persist across `pyrun` calls and are exported to the caller's namespace:

In [ ]:
# Create a persistent function
await pyrun('def fib_(n): return n if n <= 1 else fib_(n-1) + fib_(n-2)')

In [ ]:
# Use it in a subsequent call
await pyrun('[fib_(i) for i in range(10)]')

In [ ]:
# Variables with _ suffix are also exported to the caller's namespace
await pyrun('result_ = sum(range(100))')
print(f"result_ is now available here: {result_}")

## What Gets Blocked

Dangerous operations like filesystem writes, process spawning, and system modification are blocked:

In [ ]:
# Attempting to delete a file — blocked
try:
    await pyrun('import os; os.remove("/tmp/test")')
except Exception as e:
    print(f"Blocked: {type(e).__name__}: {e}")

In [ ]:
# Attempting to spawn a subprocess — blocked
try:
    await pyrun('import subprocess; subprocess.run(["ls"])')
except Exception as e:
    print(f"Blocked: {type(e).__name__}: {e}")

## Extending the Allowlist with `allow()`

Register your own functions or third-party library methods:

In [ ]:
from safepyrun import allow

# Define and allow a custom function
def greet(name): return f"Hello, {name}!"
allow('greet')

await pyrun('greet("World")')

## Write Permissions with `ok_dests`

By default, all filesystem writes are blocked. Use `ok_dests` to allow controlled writing:

In [ ]:
# Create a sandbox with write access to /tmp
pyrun_w = RunPython(ok_dests=['/tmp'])

# This works — writing to an allowed destination
await pyrun_w("Path('/tmp/safepyrun_test.txt').write_text('hello from sandbox')")

In [ ]:
# Verify the write worked
await pyrun_w("Path('/tmp/safepyrun_test.txt').read_text()")

In [ ]:
# But writing outside allowed destinations is still blocked
try:
    await pyrun_w("Path('/etc/evil.txt').write_text('bad')")
except PermissionError as e:
    print(f"Blocked: {e}")

## Async Support

The sandbox is async-native — `await`, `async for`, and `async with` all work:

In [ ]:
await pyrun('''
import asyncio
async def compute(n): return n * 10
results = await asyncio.gather(compute(1), compute(2), compute(3))
f"Async results: {results}"
''')

## Integration with Dialeng

In Dialeng, `pyrun` is registered as a **built-in LLM tool**. When the AI needs to execute Python code safely during a prompt response, it can call `pyrun` automatically — no `&` prefix needed.

The sandbox instance is created with `ok_dests=['.']` so the AI can write files relative to the current working directory.

Try creating a prompt cell and asking the AI to "use pyrun to calculate the first 20 prime numbers" to see it in action!

## Cleanup

In [ ]:
# Clean up test file
from pathlib import Path
Path('/tmp/safepyrun_test.txt').unlink(missing_ok=True)
print("Cleaned up test files")